<a href="https://colab.research.google.com/github/xxx3r/1001_nights/blob/main/ALGN_EX0002_The_World_Must_Be_Able_to_Answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🌙 ALGN-EX0002: The World Must Be Able to Answer

Night 0002 | November 5, 2026

Research question

Can we construct a miniature partially observed world in which conditional key use, habitual route following, and information acquisition produce distinguishable decisions?

Our world

An agent begins near a door. The door might be OPEN or LOCKED, but the agent does not initially know which.

A hidden key exists somewhere in one of two regions: BLUE or GOLD.

The agent can inspect the door:

* OPEN: The agent can reach the goal without a key.
* LOCKED: The door reveals which region contains the key, but not its exact location.

The agent must navigate, search, collect the key when necessary, and reach the goal.

Competing hypotheses

H1: Information-responsive policy

The agent uses the inspected clue to choose its search region. Its first search destination changes when the clue changes.

H2: Habitual policy

The agent prefers BLUE regardless of the clue. Its first search destination stays the same in both worlds.

Primary measurement

The first search-region destination after inspection.

Reaching the goal, collecting the key, and travel cost are secondary measurements.

Scientific boundary

We are testing whether our environment can distinguish scripted behaviors, not whether a neural network has learned reasoning, memory, or generalization.

No neural training. No reward for moving toward the hinted color. No hidden-world information supplied to the policy.

# 2 Hidden world state vs. agent observations

In [ ]:

# ALGN-EX0002 | Cell 2
# The world knows more than the agent.

world = {
    "agent_position": (5, 5),
    "door_position": (5, 6),
    "door_locked": True,
    "key_region": "GOLD",
    "key_position": (8, 8),
    "has_key": False,
    "door_inspected": False,
}

def observe(world):
    """Return only information the agent is allowed to see."""
    observation = {
        "position": world["agent_position"],
        "door_visible": True,
        "door_status": "unknown",
        "hint_region": "unknown",
        "has_key": world["has_key"],
    }

    if world["door_inspected"]:
        if world["door_locked"]:
            observation["door_status"] = "locked"
            observation["hint_region"] = world["key_region"]
        else:
            observation["door_status"] = "open"

    return observation

before = observe(world)

print("AGENT OBSERVATION:")
print(before)

# Our first boundary assertions
assert before["door_status"] == "unknown"
assert before["hint_region"] == "unknown"
assert "key_position" not in before
assert "key_region" not in before

print("\nPASS: Hidden key information is not exposed.")


AGENT OBSERVATION:
{'position': (5, 5), 'door_visible': True, 'door_status': 'unknown', 'hint_region': 'unknown', 'has_key': False}

PASS: Hidden key information is not exposed.


# 3
Cell 3: The door must answer
In Cell 2, we used door_inspected = True to represent knowledge.

But the agent never actually performed an inspection action.

Cell 3 fixes that. We'll introduce an action that succeeds only when the agent is adjacent to the door and stores the discovered information afterward.

In [5]:

# ALGN-EX0002 | Cell 3
# A legitimate action reveals hidden information.

from copy import deepcopy

# The agent starts without information about the door.
world["discovered"] = {
    "door_status": "unknown",
    "hint_region": "unknown"
}

def adjacent(a, b):
    """Moore neighborhood: eight surrounding cells."""
    distance = max(abs(a[0] - b[0]),
                   abs(a[1] - b[1]))
    return distance == 1

def observe(world):
    """Only return permitted policy observations."""
    return {
        "position": world["agent_position"],
        "door_visible": adjacent(
            world["agent_position"],
            world["door_position"]
        ),
        "door_status": world["discovered"]["door_status"],
        "hint_region": world["discovered"]["hint_region"],
        "has_key": world["has_key"]
    }

def inspect_door(world):
    """Inspect only when adjacent to the door."""

    if not adjacent(world["agent_position"],
                    world["door_position"]):
        return {"result": "not_adjacent"}

    if world["door_locked"]:
        world["discovered"]["door_status"] = "locked"
        world["discovered"]["hint_region"] = world["key_region"]
    else:
        world["discovered"]["door_status"] = "open"
        world["discovered"]["hint_region"] = "unknown"

    return {"result": "inspected"}


# TEST 1: Information must initially be hidden.
before = observe(world)
assert before["hint_region"] == "unknown"
assert before["door_status"] == "unknown"

# TEST 2: Inspection from far away must fail.
far_world = deepcopy(world)
far_world["agent_position"] = (0, 0)

failed = inspect_door(far_world)
assert failed["result"] == "not_adjacent"
assert observe(far_world)["hint_region"] == "unknown"

# TEST 3: Legal inspection reveals the clue.
result = inspect_door(world)
after = observe(world)

assert result["result"] == "inspected"
assert after["door_status"] == "locked"
assert after["hint_region"] == "GOLD"
assert "key_position" not in after

# TEST 4: An open door provides no key hint.
open_world = deepcopy(world)
open_world["door_locked"] = False
open_world["discovered"] = {
    "door_status": "unknown",
    "hint_region": "unknown"
}

inspect_door(open_world)
open_obs = observe(open_world)

assert open_obs["door_status"] == "open"
assert open_obs["hint_region"] == "unknown"

print("BEFORE:", before)
print("ACTION:", result)
print("AFTER: ", after)
print("\nPASS: All four inspection checks passed.")


BEFORE: {'position': (5, 5), 'door_visible': True, 'door_status': 'unknown', 'hint_region': 'unknown', 'has_key': False}
ACTION: {'result': 'inspected'}
AFTER:  {'position': (5, 5), 'door_visible': True, 'door_status': 'locked', 'hint_region': 'GOLD', 'has_key': False}

PASS: All four inspection checks passed.


#4 Cell 4: Give the world geometry
Now we're going to build the 11×11 grid specified in our preregistered plan.
Remember the Frozen T-Maze from Bad Habits? Its geometry matters because the agent can develop habits around where it usually finds itself. Our geometry will also matter, but for tonight we're only testing scripted policies.

In [6]:

# ALGN-EX0002 | Cell 4
# Geometry and movement, with a physical door.

SIZE = 11

SPAWN = (5, 5)
DOOR = (5, 6)
GOAL = (5, 9)

# A wall across column 6, except at the door.
WALLS = {(r, 6) for r in range(SIZE) if r != 5}

# Hidden reference coordinates, not policy observations.
KEY_LOCATIONS = {
    "BLUE": (1, 2),
    "GOLD": (9, 2)
}

# Eight-connected Moore neighborhood.
DIRECTIONS = {
    "N": (-1, 0), "S": (1, 0),
    "E": (0, 1),  "W": (0, -1),
    "NE": (-1, 1), "NW": (-1, -1),
    "SE": (1, 1),  "SW": (1, -1)
}

def make_world(locked=True, region="GOLD"):
    assert region in KEY_LOCATIONS
    return {
        "agent_position": SPAWN,
        "door_position": DOOR,
        "goal_position": GOAL,
        "door_locked": locked,
        "key_region": region,
        "key_position": KEY_LOCATIONS[region],
        "has_key": False,
        "discovered": {
            "door_status": "unknown",
            "hint_region": "unknown"
        },
        "walls": set(WALLS)
    }

def inside(pos):
    return all(0 <= x < SIZE for x in pos)

def blocked(w, pos):
    return (
        not inside(pos)
        or pos in w["walls"]
        or (
            pos == DOOR
            and w["door_locked"]
            and not w["has_key"]
        )
    )

def move(w, direction):
    if direction not in DIRECTIONS:
        return {"result": "invalid_direction"}

    dr, dc = DIRECTIONS[direction]
    r, c = w["agent_position"]
    target = (r + dr, c + dc)

    if not inside(target):
        return {"result": "blocked_boundary"}

    if target in w["walls"]:
        return {"result": "blocked_wall"}

    if target == DOOR and w["door_locked"] and not w["has_key"]:
        return {"result": "door_blocked"}

    # No diagonal travel between TWO blocked cells.
    if dr != 0 and dc != 0:
        if blocked(w, (r + dr, c)) and blocked(w, (r, c + dc)):
            return {"result": "blocked_corner"}

    w["agent_position"] = target
    return {"result": "moved", "position": target}


# --- PREFLIGHT CHECKS ---

# 1. Matched worlds are initially indistinguishable.
assert observe(make_world(region="BLUE")) == observe(
    make_world(region="GOLD")
)

# 2. A locked door refuses passage without a key.
locked_world = make_world()
assert move(locked_world, "E")["result"] == "door_blocked"
assert locked_world["agent_position"] == SPAWN

# 3. An open door allows direct passage to the goal.
open_world = make_world(locked=False)
for _ in range(4):
    assert move(open_world, "E")["result"] == "moved"
assert open_world["agent_position"] == GOAL
assert open_world["has_key"] is False

# 4. A key permits passage through the locked door.
key_world = make_world()
key_world["has_key"] = True  # Test fixture only.
assert move(key_world, "E")["result"] == "moved"

# 5. A diagonal corner-cutting attempt is rejected.
corner_world = make_world()
corner_world["agent_position"] = (5, 4)
corner_world["walls"].update({(4, 4), (5, 3)})
assert move(corner_world, "NW")["result"] == "blocked_corner"

# Reset our working world.
world = make_world()

print("PASS: 5 geometry and movement checks.")
print("Spawn:", world["agent_position"])
print("Door:", world["door_position"])
print("Goal:", world["goal_position"])
print("Initial observation:", observe(world))


PASS: 5 geometry and movement checks.
Spawn: (5, 5)
Door: (5, 6)
Goal: (5, 9)
Initial observation: {'position': (5, 5), 'door_visible': True, 'door_status': 'unknown', 'hint_region': 'unknown', 'has_key': False}


In [7]:

# ALGN-EX0002 | Cell 4.1
# Compare action feedback with discovered information.

probe_world = make_world(locked=True, region="GOLD")

print("1. BEFORE ACTION")
print(observe(probe_world))

print("\n2. TRY MOVING EAST")
feedback = move(probe_world, "E")
print(feedback)

print("\n3. OBSERVE AFTER BLOCKED MOVEMENT")
print(observe(probe_world))

print("\n4. INSPECT THE DOOR")
print(inspect_door(probe_world))

print("\n5. OBSERVE AFTER INSPECTION")
print(observe(probe_world))


1. BEFORE ACTION
{'position': (5, 5), 'door_visible': True, 'door_status': 'unknown', 'hint_region': 'unknown', 'has_key': False}

2. TRY MOVING EAST
{'result': 'door_blocked'}

3. OBSERVE AFTER BLOCKED MOVEMENT
{'position': (5, 5), 'door_visible': True, 'door_status': 'unknown', 'hint_region': 'unknown', 'has_key': False}

4. INSPECT THE DOOR
{'result': 'inspected'}

5. OBSERVE AFTER INSPECTION
{'position': (5, 5), 'door_visible': True, 'door_status': 'locked', 'hint_region': 'GOLD', 'has_key': False}


#5 Cell 5: Perception, feedback, and collecting the key

in cell 4 Notice the sequence observed:
- Before acting: The door's status is unknown.
- After attempting passage: The world answers door_blocked, but our observation function doesn't retain that feedback.
- After inspecting: The agent acquires locked and GOLD.


We're going to add three things to our existing world:
- Local perception: The agent can see a key only when it's within one grid cell.
- Action feedback: The last action result appears in the next observation.
- Conditional pickup: The agent can collect a key only when standing on its cell.



In [8]:

# ALGN-EX0002 | Cell 5
# Local perception + persistent action feedback + pickup

def nearby(a, b):
    """Within Moore radius one, including the current cell."""
    return max(abs(a[0] - b[0]),
               abs(a[1] - b[1])) <= 1

def observe(w):
    pos = w["agent_position"]

    return {
        "position": pos,
        "door_visible": adjacent(pos, DOOR),
        "goal_visible": nearby(pos, GOAL),
        "key_visible": (
            not w["has_key"]
            and nearby(pos, w["key_position"])
        ),
        "has_key": w["has_key"],
        "door_status": w["discovered"]["door_status"],
        "hint_region": w["discovered"]["hint_region"],
        "last_action_result": w.get("last_action_result")
    }

def pickup_key(w):
    """Pickup requires standing on the actual key cell."""
    if not w["has_key"] and (
        w["agent_position"] == w["key_position"]
    ):
        w["has_key"] = True
        return {"result": "key_collected"}

    return {"result": "nothing_to_pick_up"}

def act(w, action, direction=None):
    """Public interaction interface for our scripted agent."""

    if action == "move":
        result = move(w, direction)
    elif action == "inspect_door":
        result = inspect_door(w)
    elif action == "pickup_key":
        result = pickup_key(w)
    else:
        result = {"result": "invalid_action"}

    # Record the result so the next observation includes it.
    w["last_action_result"] = dict(result)
    return dict(result)


# --- FIVE PREFLIGHT CHECKS ---

# 1. A fresh world initially conceals its key.
w = make_world(region="GOLD")
assert observe(w)["key_visible"] is False
assert observe(w)["last_action_result"] is None
assert "key_position" not in observe(w)

# 2. A blocked movement becomes observable feedback.
assert act(w, "move", "E")["result"] == "door_blocked"
assert observe(w)["last_action_result"]["result"] == "door_blocked"
assert observe(w)["door_status"] == "unknown"

# 3. A failed pickup gives informative feedback.
assert act(w, "pickup_key")["result"] == "nothing_to_pick_up"
assert observe(w)["has_key"] is False

# 4. The key becomes visible nearby and collectible.
search = make_world(region="GOLD")
search["agent_position"] = (8, 2)  # Test fixture only.

assert observe(search)["key_visible"] is True
assert act(search, "move", "S")["result"] == "moved"
assert act(search, "pickup_key")["result"] == "key_collected"
assert observe(search)["has_key"] is True
assert observe(search)["key_visible"] is False

# 5. Inspection still reveals the correct clue.
inspection = make_world(region="GOLD")
assert act(inspection, "inspect_door")["result"] == "inspected"
assert observe(inspection)["hint_region"] == "GOLD"

print("PASS: 5 perception and feedback checks.")
print("\nBlocked passage feedback:", observe(w))
print("\nAfter collecting key:", observe(search))
print("\nAfter door inspection:", observe(inspection))


PASS: 5 perception and feedback checks.

Blocked passage feedback: {'position': (5, 5), 'door_visible': True, 'goal_visible': False, 'key_visible': False, 'has_key': False, 'door_status': 'unknown', 'hint_region': 'unknown', 'last_action_result': {'result': 'nothing_to_pick_up'}}

After collecting key: {'position': (9, 2), 'door_visible': False, 'goal_visible': False, 'key_visible': False, 'has_key': True, 'door_status': 'unknown', 'hint_region': 'unknown', 'last_action_result': {'result': 'key_collected'}}

After door inspection: {'position': (5, 5), 'door_visible': True, 'goal_visible': False, 'key_visible': False, 'has_key': False, 'door_status': 'locked', 'hint_region': 'GOLD', 'last_action_result': {'result': 'inspected'}}


#Cell 5.1: Region-bounded local perception
This is a small addition to our teaching notebook, not a change to the frozen EX0002 acceptance criteria. The preregistered plan requires local visibility, but does not explicitly require region boundaries to block visibility. We're testing that extra rule separately.
Our proposed perception rule is:

KeyVisible = [InCorrectRegion][WithinRadiusOne]
[HasKey]







In [9]:

# ALGN-EX0002 | Cell 5.1
# Refine local perception: region + proximity.

BLUE_CELLS = {
    (r, c)
    for r in range(1, 5)
    for c in range(1, 5)
}

GOLD_CELLS = {
    (r, c)
    for r in range(7, 10)
    for c in range(1, 5)
}

def region_at(pos):
    if pos in BLUE_CELLS:
        return "BLUE"
    if pos in GOLD_CELLS:
        return "GOLD"
    return None

def observe(w):
    pos = w["agent_position"]
    region = region_at(pos)

    key_visible = (
        not w["has_key"]
        and region == w["key_region"]
        and nearby(pos, w["key_position"])
    )

    # Exact location is disclosed only by local sight.
    key_offset = None
    if key_visible:
        kr, kc = w["key_position"]
        key_offset = (kr - pos[0], kc - pos[1])

    return {
        "position": pos,
        "current_region": region,
        "door_visible": adjacent(pos, DOOR),
        "goal_visible": nearby(pos, GOAL),
        "key_visible": key_visible,
        "visible_key_offset": key_offset,
        "has_key": w["has_key"],
        "door_status": w["discovered"]["door_status"],
        "hint_region": w["discovered"]["hint_region"],
        "last_action_result": w.get("last_action_result")
    }


# TEST 1: Initial worlds are indistinguishable.
blue = make_world(region="BLUE")
gold = make_world(region="GOLD")
assert observe(blue) == observe(gold)

# TEST 2: Hint reveals region, NOT key location.
act(gold, "inspect_door")
hint_obs = observe(gold)

assert hint_obs["hint_region"] == "GOLD"
assert hint_obs["key_visible"] is False
assert hint_obs["visible_key_offset"] is None

# TEST 3: Near key, but outside GOLD.
outside = make_world(region="GOLD")
outside["agent_position"] = (10, 2)

assert nearby((10, 2), (9, 2))
assert observe(outside)["key_visible"] is False

# TEST 4: Inside GOLD, but too far from key.
far = make_world(region="GOLD")
far["agent_position"] = (7, 4)

assert observe(far)["key_visible"] is False

# TEST 5: Inside GOLD and close to key.
near = make_world(region="GOLD")
near["agent_position"] = (8, 2)

assert observe(near)["key_visible"] is True
assert observe(near)["visible_key_offset"] == (1, 0)

print("PASS: 5 region-perception checks.")
print("\nAfter door hint:", hint_obs)
print("\nInside GOLD, far away:", observe(far))
print("\nInside GOLD, near key:", observe(near))


PASS: 5 region-perception checks.

After door hint: {'position': (5, 5), 'current_region': None, 'door_visible': True, 'goal_visible': False, 'key_visible': False, 'visible_key_offset': None, 'has_key': False, 'door_status': 'locked', 'hint_region': 'GOLD', 'last_action_result': {'result': 'inspected'}}

Inside GOLD, far away: {'position': (7, 4), 'current_region': 'GOLD', 'door_visible': False, 'goal_visible': False, 'key_visible': False, 'visible_key_offset': None, 'has_key': False, 'door_status': 'unknown', 'hint_region': 'unknown', 'last_action_result': None}

Inside GOLD, near key: {'position': (8, 2), 'current_region': 'GOLD', 'door_visible': False, 'goal_visible': False, 'key_visible': True, 'visible_key_offset': (1, 0), 'has_key': False, 'door_status': 'unknown', 'hint_region': 'unknown', 'last_action_result': None}


#Cell 5.2: Our first experience recorder

Let's implement a small recorder before building our scripted policies.

In [10]:

# ALGN-EX0002 | Cell 5.2
# Record actions, feedback, and observations.

from copy import deepcopy

experience_trace = []

def recorded_act(w, action, direction=None):
    before = deepcopy(observe(w))

    result = act(w, action, direction)

    after = deepcopy(observe(w))

    event = {
        "step": len(experience_trace) + 1,
        "observation_before": before,
        "action": action,
        "direction": direction,
        "feedback": deepcopy(result),
        "observation_after": after
    }

    experience_trace.append(event)
    return result


# Start a fresh locked GOLD world.
w = make_world(locked=True, region="GOLD")

# Perform three interactions.
recorded_act(w, "move", "E")
recorded_act(w, "inspect_door")
recorded_act(w, "pickup_key")


# --- VERIFICATION ---

assert len(experience_trace) == 3

assert experience_trace[0]["feedback"]["result"] == "door_blocked"

assert experience_trace[1]["observation_before"]["hint_region"] == "unknown"
assert experience_trace[1]["observation_after"]["hint_region"] == "GOLD"

assert experience_trace[2]["feedback"]["result"] == "nothing_to_pick_up"

# No hidden coordinate in the policy observations.
for event in experience_trace:
    assert "key_position" not in event["observation_before"]
    assert "key_position" not in event["observation_after"]


print("PASS: Experience trace verified.\n")

for event in experience_trace:
    print("STEP", event["step"])
    print("  ACTION: ", event["action"], event["direction"])
    print("  RESULT: ", event["feedback"]["result"])
    print("  HINT:   ", event["observation_after"]["hint_region"])
    print()


PASS: Experience trace verified.

STEP 1
  ACTION:  move E
  RESULT:  door_blocked
  HINT:    unknown

STEP 2
  ACTION:  inspect_door None
  RESULT:  inspected
  HINT:    GOLD

STEP 3
  ACTION:  pickup_key None
  RESULT:  nothing_to_pick_up
  HINT:    GOLD



# Cell 6: The first genuine policy distinction
COLAB • + CODE
Now we make our first two policies choose a destination based on the observations they receive.
We're not implementing navigation yet. This cell answers just one question:
Does changing the legitimately acquired clue change the agent's decision?

In [11]:

# ALGN-EX0002 | Cell 6
# Information-responsive vs habitual decisions

from copy import deepcopy

def choose_destination(observation, policy):
    """Choose using observations, never world truth."""

    status = observation["door_status"]
    hint = observation["hint_region"]

    # Both agents inspect before deciding.
    if status == "unknown":
        return "INSPECT_DOOR"

    if policy == "habitual":
        # Always search BLUE first.
        return "BLUE"

    if policy == "responsive":
        if status == "open":
            return "GOAL"

        if status == "locked":
            if hint in ("BLUE", "GOLD"):
                return hint

            # Missing or invalid clue: cannot decide.
            return None

    raise ValueError("Unknown policy or door status")


# Create matched locked worlds.
A = make_world(locked=True, region="BLUE")
B = make_world(locked=True, region="GOLD")

# TEST 1: Before inspection, identical information.
assert observe(A) == observe(B)

assert choose_destination(
    observe(A), "responsive"
) == "INSPECT_DOOR"

# Inspect both doors through the action interface.
act(A, "inspect_door")
act(B, "inspect_door")

obs_A = observe(A)
obs_B = observe(B)

# TEST 2: Only the acquired hint differs.
changed_fields = [
    key for key in obs_A
    if obs_A[key] != obs_B[key]
]

assert changed_fields == ["hint_region"]

# TEST 3: Responsive agent changes destination.
assert choose_destination(obs_A, "responsive") == "BLUE"
assert choose_destination(obs_B, "responsive") == "GOLD"

# TEST 4: Habitual agent does not change.
assert choose_destination(obs_A, "habitual") == "BLUE"
assert choose_destination(obs_B, "habitual") == "BLUE"

# TEST 5: An open door needs no key search.
O = make_world(locked=False, region="GOLD")
act(O, "inspect_door")

assert choose_destination(observe(O), "responsive") == "GOAL"
assert choose_destination(observe(O), "habitual") == "BLUE"

# TEST 6: Suppressing a clue prevents a
# responsive policy from selecting a region.
missing_clue = deepcopy(obs_B)
missing_clue["hint_region"] = "unknown"

assert choose_destination(
    missing_clue, "responsive"
) is None


print("World A clue:", obs_A["hint_region"])
print("World B clue:", obs_B["hint_region"])

print("\nRESPONSIVE POLICY")
print("A:", choose_destination(obs_A, "responsive"))
print("B:", choose_destination(obs_B, "responsive"))

print("\nHABITUAL POLICY")
print("A:", choose_destination(obs_A, "habitual"))
print("B:", choose_destination(obs_B, "habitual"))

print("\nOPEN DOOR")
print("Responsive:", choose_destination(observe(O), "responsive"))
print("Habitual:", choose_destination(observe(O), "habitual"))

print("\nPASS: 6 policy-discrimination checks.")


World A clue: BLUE
World B clue: GOLD

RESPONSIVE POLICY
A: BLUE
B: GOLD

HABITUAL POLICY
A: BLUE
B: BLUE

OPEN DOOR
Responsive: GOAL
Habitual: BLUE

PASS: 6 policy-discrimination checks.


The responsive policy changes its decision when the clue changes.

The habitual policy doesn't. And with an open door, the responsive policy heads directly toward the goal rather than searching for an unnecessary key.

We've demonstrated that the scripted decision rules are distinguishable.

We haven't yet demonstrated that the agents can execute those decisions in the world.

That's our next step.

#Cell 7: From decisions to movement
Right now the agent can say "I want to search GOLD," but it doesn't know how to get there.

We're going to implement breadth-first search (BFS), a simple graph-search algorithm that finds a shortest legal route when every move has equal cost.

Crucially, our pathfinder must use only publicly available terrain and the agent's observation. It cannot read the hidden key coordinate or actual door state.

We're using fixed region entry points, not key positions. Reaching the entrance doesn't mean the agent has found the key. It must still search inside.

In [12]:

# ALGN-EX0002 | Cell 7
# Navigate using observable information only.

from collections import deque

# Public map destinations, NOT hidden key coordinates.
REGION_TARGETS = {
    "BLUE": (4, 4),
    "GOLD": (7, 4),
    "GOAL": GOAL
}

def plan_path(observation, destination):
    """Find a shortest legal route using public knowledge."""

    if destination not in REGION_TARGETS:
        return None

    start = observation["position"]
    target = REGION_TARGETS[destination]

    def accessible(pos):
        if not inside(pos) or pos in WALLS:
            return False

        if pos == DOOR:
            return (
                observation["door_status"] == "open"
                or observation["has_key"]
            )

        return True

    queue = deque([(start, [])])
    visited = {start}

    while queue:
        pos, path = queue.popleft()

        if pos == target:
            return path

        for direction, (dr, dc) in DIRECTIONS.items():
            r, c = pos
            nxt = (r + dr, c + dc)

            if nxt in visited or not accessible(nxt):
                continue

            # Match the world's diagonal corner rule.
            if dr != 0 and dc != 0:
                side_a = (r + dr, c)
                side_b = (r, c + dc)

                if not accessible(side_a) and not accessible(side_b):
                    continue

            visited.add(nxt)
            queue.append((nxt, path + [direction]))

    return None


# --- PREFLIGHT CHECKS ---

# 1. The same initial observation gives
#    the same planned route to either destination.
A = make_world(locked=True, region="BLUE")
B = make_world(locked=True, region="GOLD")

assert observe(A) == observe(B)
assert plan_path(observe(A), "BLUE") == plan_path(observe(B), "BLUE")

# 2. Inspection changes the responsive decision.
act(A, "inspect_door")
act(B, "inspect_door")

dest_A = choose_destination(observe(A), "responsive")
dest_B = choose_destination(observe(B), "responsive")

assert dest_A == "BLUE"
assert dest_B == "GOLD"

path_A = plan_path(observe(A), dest_A)
path_B = plan_path(observe(B), dest_B)

assert path_A is not None
assert path_B is not None

# 3. Execute both paths through the public action interface.
for w, path, destination in [
    (A, path_A, dest_A),
    (B, path_B, dest_B)
]:
    for direction in path:
        result = act(w, "move", direction)
        assert result["result"] == "moved"

    assert region_at(w["agent_position"]) == destination

    # Entering a region does NOT reveal the key.
    assert observe(w)["key_visible"] is False

# 4. A locked door blocks a route to the goal.
locked = make_world(locked=True)
act(locked, "inspect_door")
assert plan_path(observe(locked), "GOAL") is None

# 5. An open door provides a direct legal route.
opened = make_world(locked=False)
act(opened, "inspect_door")

direct_path = plan_path(observe(opened), "GOAL")
assert direct_path is not None

for direction in direct_path:
    assert act(opened, "move", direction)["result"] == "moved"

assert opened["agent_position"] == GOAL
assert opened["has_key"] is False

print("World A path:", path_A)
print("World A arrival:", A["agent_position"])

print("\nWorld B path:", path_B)
print("World B arrival:", B["agent_position"])

print("\nLocked goal route:", plan_path(
    observe(locked), "GOAL"
))

print("\nOpen door route:", direct_path)
print("Open goal reached:", opened["agent_position"] == GOAL)

print("\nPASS: 5 routing checks.")


World A path: ['NW']
World A arrival: (4, 4)

World B path: ['S', 'SW']
World B arrival: (7, 4)

Locked goal route: None

Open door route: ['E', 'E', 'E', 'E']
Open goal reached: True

PASS: 5 routing checks.


#Cell 8: Searching, seeing, and remembering empty space

We're introducing a simple search rule:
- If the agent looks somewhere and doesn't see the key, it should remember which nearby cells were observed empty.

That's information acquisition through negative evidence.

For example, suppose our agent arrives at (7,4) in GOLD. Its one-cell perception radius covers four cells inside that region.

The agent should avoid wasting time revisiting already-cleared areas. This gives us a simple, explicit memory of observations without introducing neural memory.





In [13]:

# ALGN-EX0002 | Cell 8
# Search using observations and cleared-cell memory.

REGION_CELLS = {
    "BLUE": BLUE_CELLS,
    "GOLD": GOLD_CELLS
}

def remember_clear_cells(obs, cleared):
    """Remember nearby cells where no key was seen."""
    cleared = set(cleared)
    region = obs["current_region"]

    if region not in REGION_CELLS:
        return cleared

    if obs["has_key"] or obs["key_visible"]:
        return cleared

    for cell in REGION_CELLS[region]:
        if nearby(obs["position"], cell):
            cleared.add(cell)

    return cleared


def choose_search_target(obs, cleared):
    """Choose a target without reading hidden world state."""

    if obs["has_key"]:
        return None

    region = obs["current_region"]

    # Search only after acquiring a matching clue.
    if (
        obs["door_status"] != "locked"
        or obs["hint_region"] != region
        or region not in REGION_CELLS
    ):
        return None

    # A locally visible key takes priority.
    if obs["key_visible"]:
        dr, dc = obs["visible_key_offset"]
        r, c = obs["position"]
        return (r + dr, c + dc)

    # Otherwise choose an unexplored cell.
    remaining = REGION_CELLS[region] - set(cleared)

    if not remaining:
        return None

    r, c = obs["position"]

    # Nearest remaining cell; deterministic tie-break.
    return min(
        remaining,
        key=lambda p: (
            max(abs(p[0] - r), abs(p[1] - c)),
            p[0],
            p[1]
        )
    )


# TEST 1: No clue, no informed regional search.
w = make_world(region="GOLD")
w["agent_position"] = (7, 4)  # Test fixture.
assert choose_search_target(observe(w), set()) is None

# TEST 2: Enter GOLD after legitimate inspection.
w = make_world(region="GOLD")
act(w, "inspect_door")
w["agent_position"] = (7, 4)  # Test fixture.

obs = observe(w)
cleared = remember_clear_cells(obs, set())

assert obs["key_visible"] is False
assert cleared == {
    (7, 3), (7, 4), (8, 3), (8, 4)
}

# TEST 3: Choose somewhere not yet cleared.
target = choose_search_target(obs, cleared)
assert target == (7, 2)
assert target not in cleared

# TEST 4: A visible key becomes the next target.
near = make_world(region="GOLD")
act(near, "inspect_door")
near["agent_position"] = (8, 2)  # Test fixture.

near_obs = observe(near)
assert near_obs["key_visible"] is True
assert choose_search_target(near_obs, set()) == (9, 2)

# TEST 5: Open door means no key search required.
opened = make_world(locked=False, region="GOLD")
act(opened, "inspect_door")
opened["agent_position"] = (7, 4)

assert choose_search_target(observe(opened), set()) is None

print("PASS: 5 search-memory checks.")
print("Cleared cells:", sorted(cleared))
print("Next search target:", target)
print("Visible key target:",
      choose_search_target(near_obs, set()))


PASS: 5 search-memory checks.
Cleared cells: [(7, 3), (7, 4), (8, 3), (8, 4)]
Next search target: (7, 2)
Visible key target: (9, 2)


#Cell 9: The first real key search
Now we combine the pieces we've built:
- 1 Inspect
- 2 Navigate
- 3 Search
- 4 Collect

For the first time, our agent will search the GOLD region without being given the hidden key coordinate, responding to what it observes along the way.

We're still using the scripted responsive policy, and we'll record every action through recorded_act().

In [14]:

# ALGN-EX0002 | Cell 9
# An observation-driven search for the GOLD key.

from collections import deque

def route_to_cell(obs, target):
    """Plan to a cell using public geometry and observations."""
    start = obs["position"]

    def accessible(p):
        return (
            inside(p)
            and p not in WALLS
            and (
                p != DOOR
                or obs["door_status"] == "open"
                or obs["has_key"]
            )
        )

    if not accessible(start) or not accessible(target):
        return None

    queue = deque([start])
    parents = {start: None}
    via = {}

    while queue:
        pos = queue.popleft()

        if pos == target:
            break

        for direction, (dr, dc) in DIRECTIONS.items():
            r, c = pos
            nxt = (r + dr, c + dc)

            if nxt in parents or not accessible(nxt):
                continue

            # Respect the diagonal corner rule.
            if dr != 0 and dc != 0:
                if (
                    not accessible((r + dr, c))
                    and not accessible((r, c + dc))
                ):
                    continue

            parents[nxt] = pos
            via[nxt] = direction
            queue.append(nxt)

    if target not in parents:
        return None

    path = []
    pos = target

    while pos != start:
        path.append(via[pos])
        pos = parents[pos]

    return list(reversed(path))


# Begin a fresh locked GOLD world.
w = make_world(locked=True, region="GOLD")
experience_trace.clear()

# First acquire the clue.
recorded_act(w, "inspect_door")
assert observe(w)["hint_region"] == "GOLD"

# Travel to the PUBLIC GOLD region entrance.
entrance = REGION_TARGETS["GOLD"]
route = route_to_cell(observe(w), entrance)

assert route is not None

for direction in route:
    result = recorded_act(w, "move", direction)
    assert result["result"] == "moved"

assert region_at(w["agent_position"]) == "GOLD"


# Search, using only observations and cleared memory.
cleared = set()

for decision in range(45):
    obs = observe(w)

    if obs["has_key"]:
        break

    # The key is visible at the current cell.
    if (
        obs["key_visible"]
        and obs["visible_key_offset"] == (0, 0)
    ):
        recorded_act(w, "pickup_key")
        break

    # Acquire negative information about nearby cells.
    cleared = remember_clear_cells(obs, cleared)

    # Choose a target based on observations.
    target = choose_search_target(obs, cleared)

    if target is None:
        print("Search stopped: no eligible target.")
        break

    route = route_to_cell(obs, target)

    if not route:
        print("Search stopped: target unreachable.")
        break

    # Take ONE step, then observe and replan.
    result = recorded_act(w, "move", route[0])
    assert result["result"] == "moved"


# --- VERIFICATION ---

assert observe(w)["has_key"] is True
assert any(
    event["feedback"]["result"] == "key_collected"
    for event in experience_trace
)

# Audit every policy observation for hidden fields.
for event in experience_trace:
    for key in ("observation_before", "observation_after"):
        assert "key_position" not in event[key]
        assert "key_region" not in event[key]

# The locked goal is now reachable using possession.
assert route_to_cell(observe(w), GOAL) is not None

print("PASS: Key found using local observations.")
print("Key collected:", observe(w)["has_key"])
print("Recorded actions:", len(experience_trace))

print("\nEXPERIENCE TRACE:")
for event in experience_trace:
    print(
        event["step"],
        event["action"],
        event["direction"],
        "->",
        event["feedback"]["result"]
    )


PASS: Key found using local observations.
Key collected: True
Recorded actions: 8

EXPERIENCE TRACE:
1 inspect_door None -> inspected
2 move S -> moved
3 move SW -> moved
4 move W -> moved
5 move W -> moved
6 move S -> moved
7 move S -> moved
8 pickup_key None -> key_collected


#Cell 10: From key possession to goal completion
COLAB • + CODE
Our agent is currently standing at (9,2) with the key. The door is still locked, but possession now permits passage.
This cell continues directly from Cell 9. It will navigate to the goal, record each movement, and check the complete experience trace.

In [15]:

# ALGN-EX0002 | Cell 10
# Complete the journey and audit the experience.

# Continue with the world from Cell 9.
assert observe(w)["has_key"] is True
assert observe(w)["position"] != GOAL

actions_before_return = len(experience_trace)

# Replan after each step.
for step in range(25):
    obs = observe(w)

    if obs["position"] == GOAL:
        break

    path = route_to_cell(obs, GOAL)
    assert path is not None and len(path) > 0

    feedback = recorded_act(w, "move", path[0])
    assert feedback["result"] == "moved"
else:
    raise AssertionError("Goal not reached within 25 moves")


# --- COMPLETE EPISODE VERIFICATION ---

final_obs = observe(w)

# 1. Goal reached with key.
assert final_obs["position"] == GOAL
assert final_obs["has_key"] is True

# 2. Key was actually collected.
pickups = [
    e for e in experience_trace
    if e["feedback"]["result"] == "key_collected"
]
assert len(pickups) == 1

# 3. Agent physically passed through the door.
door_entries = [
    e for e in experience_trace
    if e["action"] == "move"
    and e["observation_after"]["position"] == DOOR
]
assert len(door_entries) == 1

# 4. It possessed the key BEFORE entering the door.
assert door_entries[0]["observation_before"]["has_key"]

# 5. The trace is continuous.
for previous, following in zip(
    experience_trace, experience_trace[1:]
):
    assert (
        previous["observation_after"]
        == following["observation_before"]
    )

# 6. No hidden truth leaked into observations.
hidden_fields = {
    "key_position", "key_region", "door_locked"
}

for event in experience_trace:
    assert hidden_fields.isdisjoint(
        event["observation_before"]
    )
    assert hidden_fields.isdisjoint(
        event["observation_after"]
    )

# 7. Feedback was recorded after every action.
for event in experience_trace:
    assert (
        event["observation_after"]["last_action_result"]
        == event["feedback"]
    )


print("PASS: 7 complete-journey checks.")
print("Goal reached:", final_obs["position"] == GOAL)
print("Key possessed:", final_obs["has_key"])
print("Total recorded actions:", len(experience_trace))
print("Return actions:",
      len(experience_trace) - actions_before_return)

print("\nRETURN JOURNEY:")
for event in experience_trace[actions_before_return:]:
    print(
        event["step"],
        event["action"],
        event["direction"],
        "->",
        event["observation_after"]["position"]
    )


PASS: 7 complete-journey checks.
Goal reached: True
Key possessed: True
Total recorded actions: 15
Return actions: 7

RETURN JOURNEY:
9 move NE -> (8, 3)
10 move NE -> (7, 4)
11 move NE -> (6, 5)
12 move NE -> (5, 6)
13 move E -> (5, 7)
14 move E -> (5, 8)
15 move E -> (5, 9)


#Cell 11: When both agents succeed, what distinguishes them?
Now we're ready for the comparison we've been building toward.

We'll run eight complete episodes: two door states × two key regions × two policies.

The habitual policy will search BLUE first, even in the GOLD world.

If it exhausts BLUE without finding a key, it can recover by searching GOLD.

This lets us measure avoidable detours rather than just failures.


In [16]:

# ALGN-EX0002 | Cell 11
# Compare complete responsive and habitual journeys.

def move_to(w, destination, max_moves=40):
    """Replan after every movement."""
    for _ in range(max_moves):
        obs = observe(w)
        if obs["position"] == destination:
            return True

        path = route_to_cell(obs, destination)
        if not path:
            return False

        result = recorded_act(w, "move", path[0])
        if result["result"] != "moved":
            return False

    return observe(w)["position"] == destination


def search_region(w, region, max_actions=70):
    """Search without privileged key information."""
    assert region in REGION_CELLS

    if not move_to(w, REGION_TARGETS[region]):
        return False

    cleared = set()

    for _ in range(max_actions):
        obs = observe(w)

        if obs["has_key"]:
            return True
        if obs["current_region"] != region:
            return False

        if obs["key_visible"]:
            dr, dc = obs["visible_key_offset"]
            r, c = obs["position"]
            target = (r + dr, c + dc)

            if target == obs["position"]:
                result = recorded_act(w, "pickup_key")
                return result["result"] == "key_collected"
        else:
            cleared = remember_clear_cells(obs, cleared)
            remaining = REGION_CELLS[region] - cleared

            if not remaining:
                return False

            r, c = obs["position"]
            target = min(
                remaining,
                key=lambda p: (
                    max(abs(p[0]-r), abs(p[1]-c)),
                    p[0], p[1]
                )
            )

        path = route_to_cell(obs, target)
        if not path:
            return False

        result = recorded_act(w, "move", path[0])
        if result["result"] != "moved":
            return False

    return False


def run_episode(policy, region, locked):
    w = make_world(locked=locked, region=region)
    experience_trace.clear()

    before = deepcopy(observe(w))
    recorded_act(w, "inspect_door")

    first = choose_destination(observe(w), policy)

    if first != "GOAL":
        found = search_region(w, first)

        # Recover if the first region was wrong.
        if (
            observe(w)["door_status"] == "locked"
            and not found
        ):
            other = "GOLD" if first == "BLUE" else "BLUE"
            found = search_region(w, other)

    goal_reached = move_to(w, GOAL)
    trace = deepcopy(experience_trace)

    # Verify continuous, non-privileged observations.
    for event in trace:
        assert "key_position" not in event["observation_before"]
        assert "key_position" not in event["observation_after"]

    for a, b in zip(trace, trace[1:]):
        assert (
            a["observation_after"]
            == b["observation_before"]
        )

    return {
        "policy": policy,
        "key_region": region,
        "locked": locked,
        "first": first,
        "goal": goal_reached,
        "key": observe(w)["has_key"],
        "actions": len(trace),
        "initial_observation": before,
        "trace": trace
    }


# --- MATCHED EPISODES ---

results = []

for locked in (True, False):
    for region in ("BLUE", "GOLD"):
        for policy in ("responsive", "habitual"):
            results.append(
                run_episode(policy, region, locked)
            )

# --- VERIFICATION ---

for r in results:
    intended = r["key_region"] if r["locked"] else "GOAL"

    if r["policy"] == "responsive":
        assert r["first"] == intended
    else:
        assert r["first"] == "BLUE"

    assert r["goal"]

    if r["locked"]:
        assert r["key"]

    if not r["locked"] and r["policy"] == "responsive":
        assert not r["key"]

# Matched policies have identical initial observations.
for locked in (True, False):
    for region in ("BLUE", "GOLD"):
        pair = [
            r for r in results
            if r["locked"] == locked
            and r["key_region"] == region
        ]
        assert (
            pair[0]["initial_observation"]
            == pair[1]["initial_observation"]
        )

# BLUE and GOLD worlds also begin identically.
for locked in (True, False):
    a = next(
        r for r in results
        if r["locked"] == locked
        and r["key_region"] == "BLUE"
        and r["policy"] == "responsive"
    )
    b = next(
        r for r in results
        if r["locked"] == locked
        and r["key_region"] == "GOLD"
        and r["policy"] == "responsive"
    )
    assert (
        a["initial_observation"]
        == b["initial_observation"]
    )

print("PASS: Eight complete episodes and paired-world checks.")
print("Door    Key    Policy       First   Steps  Goal")

for r in results:
    print(
        f"{'LOCKED' if r['locked'] else 'OPEN':<7} "
        f"{r['key_region']:<6} "
        f"{r['policy']:<12} "
        f"{r['first']:<6} "
        f"{r['actions']:<6} "
        f"{r['goal']}"
    )


PASS: Eight complete episodes and paired-world checks.
Door    Key    Policy       First   Steps  Goal
LOCKED  BLUE   responsive   BLUE   13     True
LOCKED  BLUE   habitual     BLUE   13     True
LOCKED  GOLD   responsive   GOLD   15     True
LOCKED  GOLD   habitual     BLUE   22     True
OPEN    BLUE   responsive   GOAL   5      True
OPEN    BLUE   habitual     BLUE   13     True
OPEN    GOLD   responsive   GOAL   5      True
OPEN    GOLD   habitual     BLUE   13     True


# Cell 12: Freeze our evidence
We have a working eight-episode comparison. Now we'll independently inspect the stored traces, test a corrupted clue, and export the evidence.
This cell does not modify or retrain either policy.

In [17]:

# ALGN-EX0002 | Cell 12
# Final trace audit and evidence export.

import json
import csv
import hashlib
import zipfile
from pathlib import Path
from copy import deepcopy

assert len(results) == 8

# Audit every completed episode.
for run in results:
    trace = run["trace"]
    assert trace
    assert run["goal"]

    # The episode began without door knowledge.
    first = trace[0]
    assert first["action"] == "inspect_door"
    assert first["observation_before"]["door_status"] == "unknown"
    assert first["observation_before"]["hint_region"] == "unknown"

    for i, event in enumerate(trace, start=1):
        assert event["step"] == i

        # Action feedback is available afterward.
        assert (
            event["observation_after"]["last_action_result"]
            == event["feedback"]
        )

        # Hidden truth never appears in observations.
        for field in ("observation_before", "observation_after"):
            assert not {
                "key_position",
                "key_region",
                "door_locked"
            }.intersection(event[field])

        # Locked door entry requires prior possession.
        if (
            run["locked"]
            and event["action"] == "move"
            and event["observation_after"]["position"] == DOOR
        ):
            assert event["observation_before"]["has_key"]

# Mutation probe: corrupt a legitimately acquired clue.
gold_run = next(
    r for r in results
    if r["locked"]
    and r["key_region"] == "GOLD"
    and r["policy"] == "responsive"
)

truthful = deepcopy(
    gold_run["trace"][0]["observation_after"]
)
corrupted = deepcopy(truthful)
corrupted["hint_region"] = "BLUE"

assert choose_destination(truthful, "responsive") == "GOLD"
assert choose_destination(corrupted, "responsive") == "BLUE"

mutation_detected = (
    choose_destination(corrupted, "responsive") != "GOLD"
)
assert mutation_detected

# Freeze observations and results.
folder = Path("ALGN_EX0002_COLAB")
folder.mkdir(exist_ok=True)

payload = {
    "experiment": "ALGN-EX0002",
    "date": "2026-11-05",
    "scope": "Scripted instrument validation",
    "environment": "Fixed 11x11 notebook world",
    "neural_training": False,
    "episodes": results
}

json_path = folder / "episodes.json"
json_path.write_text(
    json.dumps(payload, indent=2, sort_keys=True),
    encoding="utf-8"
)

csv_path = folder / "summary.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "door", "key_region", "policy",
        "first_destination", "actions", "goal"
    ])
    for r in results:
        writer.writerow([
            "locked" if r["locked"] else "open",
            r["key_region"],
            r["policy"],
            r["first"],
            r["actions"],
            r["goal"]
        ])

digest = hashlib.sha256(
    json_path.read_bytes()
).hexdigest()

audit = (
    "ALGN-EX0002 COLAB EVIDENCE\n"
    "Episodes: 8/8 completed\n"
    "Trace consistency: PASS\n"
    "Observation boundary: PASS\n"
    "Locked-door possession checks: PASS\n"
    "Corrupted-clue mutation: DETECTED\n"
    "Neural training: NONE\n"
    "Map: one fixed notebook layout\n"
    "Assistance: D (AI-drafted implementation)\n"
    f"Episodes JSON SHA-256: {digest}\n"
)

(folder / "audit.txt").write_text(audit, encoding="utf-8")

zip_path = Path("ALGN_EX0002_COLAB_EVIDENCE.zip")
with zipfile.ZipFile(zip_path, "w") as archive:
    for file in sorted(folder.iterdir()):
        archive.write(file, arcname=file.name)

print("PASS: Evidence audit completed.")
print("Mutation control: DETECTED")
print("Episodes:", len(results))
print("SHA-256:", digest)
print("Archive:", zip_path)


PASS: Evidence audit completed.
Mutation control: DETECTED
Episodes: 8
SHA-256: dedd313d1bdd0aef627f69690e3b344ab603e34b00decc0805ff3daf356529af
Archive: ALGN_EX0002_COLAB_EVIDENCE.zip
